# Mat & Comp – Lección 4: El algoritmo de Euclides
## Máximo común divisor, baldosas cuadradas y el algoritmo más antiguo del mundo

**Curso:** Matemáticas y Computación · **Duración sugerida:** 2 sesiones de 50 minutos

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Calcular el **máximo común divisor** (MCD) de dos números listando divisores, y entender por qué ese método es lento.
2. Programar el **algoritmo de Euclides** con restas y con residuos (`%`), usando la asignación doble `a, b = b, a % b`.
3. **Visualizar** el algoritmo como un rectángulo que se llena de cuadrados.
4. Usar el MCD para **simplificar fracciones** y calcular el **mínimo común múltiplo** (MCM).
5. Descubrir con la computadora qué números hacen trabajar más a Euclides (¡aparece Fibonacci!) y con qué probabilidad dos números no comparten divisores.

## 🔍 El problema del piso

Un salón mide **84 × 60 decímetros**. Queremos cubrir el piso con **baldosas cuadradas, todas iguales**, sin cortar ninguna.

- ¿De qué tamaño puede ser la baldosa? (1 × 1 funciona, pero son 5040 baldosas…)
- **¿Cuál es la baldosa cuadrada más grande posible?**

✏️ *Antes de seguir, intenta resolverlo con papel. ¿Sirve una baldosa de 10 × 10? ¿Y de 6 × 6? ¿Y de 12 × 12?*

Este problema tiene una solución que Euclides escribió hace unos **2300 años** en su libro *Elementos*. Es uno de los algoritmos más antiguos que todavía se usan: está dentro de tu celular cada vez que haces una compra segura en internet.

## 📚 El concepto: divisores comunes

El **máximo común divisor** de a y b, escrito **MCD(a, b)**, es el número más grande que divide a los dos.

Ejemplo con 12 y 18:
- Divisores de 12: 1, 2, 3, 4, **6**, 12
- Divisores de 18: 1, 2, 3, **6**, 9, 18
- Comunes: 1, 2, 3, 6 → **MCD(12, 18) = 6**

En el problema del piso, el lado de la baldosa debe dividir a 84 **y** a 60. La más grande es la del MCD.

Si MCD(a, b) = 1 decimos que a y b son **coprimos** (no comparten ningún divisor aparte del 1).

## 1. El método lento: listar divisores

In [ ]:
def divisores(n):
    """Lista de divisores de n (de la Lección 3)."""
    return [d for d in range(1, n + 1) if n % d == 0]

def mcd_lento(a, b):
    """MCD buscando el mayor divisor común."""
    comunes = [d for d in divisores(a) if b % d == 0]
    return max(comunes)

print("Divisores de 84:", divisores(84))
print("Divisores de 60:", divisores(60))
print("MCD(84, 60) =", mcd_lento(84, 60))

La baldosa más grande es de **12 × 12**: caben 84/12 = 7 a lo largo y 60/12 = 5 a lo ancho, en total 35 baldosas.

¿Y con números grandes? Probemos cuánto tarda.

In [ ]:
import time

inicio = time.time()
print(mcd_lento(7654321, 12345678))
print("Tiempo:", round(time.time() - inicio, 2), "segundos")

Funciona, pero para números de 8 cifras ya hace **unos 20 millones de divisiones** (revisa cada posible divisor). Para números de 300 cifras (los que se usan en criptografía) este método no terminaría ni en la edad del universo. Necesitamos una idea mejor.

## 2. La idea de Euclides: restar

**Observación clave:** si un número d divide a **a** y a **b**, también divide a su diferencia **a − b**.

Así que MCD(84, 60) = MCD(60, 84 − 60) = MCD(60, 24). ¡Los números se hicieron más pequeños y el MCD sigue igual!

Repetimos: siempre restamos el pequeño al grande, hasta que los dos sean iguales.

In [ ]:
def mcd_restas(a, b):
    """Euclides con restas, mostrando cada paso."""
    while a != b:
        print(f"MCD({a}, {b})")
        if a > b:
            a = a - b
        else:
            b = b - a
    print(f"MCD({a}, {b})  → los dos son iguales")
    return a

mcd_restas(84, 60)

### Dibujemos el algoritmo

Restar b de a es como **recortar un cuadrado de lado b** del rectángulo a × b. Lo que queda es un rectángulo más pequeño, y repetimos. El último cuadrado que encaja perfecto es la baldosa del MCD.

In [ ]:
import matplotlib.pyplot as plt

def dibujar_euclides(a, b):
    """Llena un rectángulo a × b con cuadrados, como hace Euclides."""
    fig, ax = plt.subplots(figsize=(7, 7 * b / a if a >= b else 7))
    x, y = 0, 0
    ancho, alto = a, b
    colores = ["#8ecae6", "#ffb703", "#90be6d", "#f28482", "#cdb4db", "#bde0fe"]
    paso = 0
    while ancho > 0 and alto > 0:
        lado = min(ancho, alto)
        color = colores[paso % len(colores)]
        if ancho >= alto:                      # cuadrados en fila, de izquierda a derecha
            veces = ancho // alto
            for k in range(veces):
                ax.add_patch(plt.Rectangle((x + k * lado, y), lado, lado, fc=color, ec="black"))
            x = x + veces * lado
            ancho = ancho - veces * lado
        else:                                  # cuadrados en columna, de abajo hacia arriba
            veces = alto // ancho
            for k in range(veces):
                ax.add_patch(plt.Rectangle((x, y + k * lado), lado, lado, fc=color, ec="black"))
            y = y + veces * lado
            alto = alto - veces * lado
        print(f"{veces} cuadrado(s) de lado {lado}")
        paso = paso + 1
    ax.set_xlim(0, a); ax.set_ylim(0, b)
    ax.set_aspect("equal")
    ax.set_title(f"Euclides en un rectángulo {a} × {b}: el último cuadrado mide {lado}")
    plt.show()

dibujar_euclides(84, 60)

**¿Qué pasa si…?**
- …dibujas un rectángulo de **13 × 8**? ¿Y de **21 × 13**? ¿Qué tienen de especial esos números (pista: Lección 2)?
- …dibujas **100 × 1**? ¿Cuántos cuadrados aparecen?
- …dibujas dos números **coprimos**, como 17 × 5? ¿De qué tamaño es el último cuadrado?

In [ ]:
# Prueba aquí tus rectángulos
dibujar_euclides(13, 8)

## 3. Más rápido: restar muchas veces de golpe = residuo

Con 1000 y 3 tendríamos que restar 3 unas 333 veces. Pero restar b de a "todas las veces que se pueda" es justamente lo que hace el **residuo**: lo que sobra es `a % b`.

**Algoritmo de Euclides:**
- Si b = 0, el MCD es a.
- Si no, MCD(a, b) = MCD(b, a % b).

En Python podemos actualizar dos variables a la vez con **asignación doble**: `a, b = b, a % b`.

In [ ]:
# La asignación doble: Python calcula todo lo de la derecha antes de guardar
x, y = 5, 9
print(x, y)
x, y = y, x      # ¡intercambio en una línea!
print(x, y)

In [ ]:
def mcd(a, b):
    """Algoritmo de Euclides con residuos."""
    while b != 0:
        a, b = b, a % b
    return a

def mcd_explicado(a, b):
    """Igual que mcd, pero muestra cada división."""
    pasos = 0
    while b != 0:
        print(f"{a} = {a // b} · {b} + {a % b}")
        a, b = b, a % b
        pasos = pasos + 1
    print("MCD =", a, "en", pasos, "pasos")
    return a

mcd_explicado(84, 60)
print()
mcd_explicado(1234567, 7654321)

In [ ]:
# Comparemos la velocidad con el método lento
inicio = time.time()
print(mcd(1234567, 7654321))
print("Tiempo:", time.time() - inicio, "segundos")

# ¡Hasta con números de 60 cifras!
a = 2**200 - 1
b = 3**120 - 1
print("MCD de dos números gigantes:", mcd(a, b))

Python ya trae esta función: `math.gcd` (del inglés *greatest common divisor*). Comprobemos que nuestra versión coincide.

In [ ]:
import math

for a, b in [(84, 60), (17, 5), (1071, 462), (2**20, 6**8)]:
    print(a, b, mcd(a, b), math.gcd(a, b))

## 4. ¿Qué números hacen trabajar más a Euclides?

Contemos los **pasos** (divisiones) que necesita Euclides para cada pareja (a, b) con a, b entre 1 y 100, y pintémoslos como un mapa de calor.

In [ ]:
import numpy as np

def pasos_euclides(a, b):
    pasos = 0
    while b != 0:
        a, b = b, a % b
        pasos = pasos + 1
    return pasos

N = 100
tabla = np.zeros((N + 1, N + 1))
for a in range(1, N + 1):
    for b in range(1, N + 1):
        tabla[b, a] = pasos_euclides(a, b)

plt.figure(figsize=(7, 6))
plt.imshow(tabla, origin="lower", cmap="viridis")
plt.colorbar(label="pasos")
plt.xlabel("a"); plt.ylabel("b")
plt.title("Pasos del algoritmo de Euclides para MCD(a, b)")
plt.show()

In [ ]:
# ¿Quién es el campeón de pasos? Buscamos el récord para cada cantidad de pasos
record = 0
for a in range(1, 1000):
    for b in range(1, a):
        p = pasos_euclides(a, b)
        if p > record:
            record = p
            print(f"Nuevo récord: MCD({a}, {b}) necesita {p} pasos")

¿Reconoces los números de los récords? **1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987…** ¡Son los números de **Fibonacci**!

Esto lo demostró Gabriel Lamé en 1844: las parejas de Fibonacci consecutivos son el **peor caso** de Euclides. Aun así, el número de pasos crece muy despacio: para números de 100 cifras, Euclides nunca necesita más de unos 480 pasos.

**¿Qué pasa si…?**
- …dibujas `dibujar_euclides(89, 55)`? ¿Cuántos cuadrados de cada tamaño aparecen?
- …miras en el mapa de calor la **diagonal** a = b? ¿Y la recta a = 2b? ¿Por qué tienen tan pocos pasos?

## 5. ¿Cuántas parejas son coprimas?

Pintemos de negro las parejas (a, b) con MCD = 1 y de blanco las demás.

In [ ]:
N = 60
coprimos = np.zeros((N + 1, N + 1))
for a in range(1, N + 1):
    for b in range(1, N + 1):
        if mcd(a, b) == 1:
            coprimos[b, a] = 1

plt.figure(figsize=(7, 7))
plt.imshow(coprimos, origin="lower", cmap="Greys")
plt.xlabel("a"); plt.ylabel("b")
plt.title("Parejas coprimas (negro = MCD 1)")
plt.show()

Fíjate en las líneas blancas: las filas y columnas pares tienen huecos cada 2, las múltiplos de 3 cada 3… Es la criba de la Lección 3 en dos dimensiones.

Ahora una pregunta de probabilidad: **si elijo dos números al azar, ¿qué tan probable es que sean coprimos?**

In [ ]:
import random

def probabilidad_coprimos(N, intentos):
    exitos = 0
    for _ in range(intentos):
        a = random.randint(1, N)
        b = random.randint(1, N)
        if mcd(a, b) == 1:
            exitos = exitos + 1
    return exitos / intentos

for intentos in [100, 1000, 10000, 100000]:
    print(intentos, "intentos →", probabilidad_coprimos(10**6, intentos))

In [ ]:
# El valor exacto (cuando N es muy grande) es 6 / π²
print("6 / π² =", 6 / math.pi**2)

# ¡Así que podemos estimar π usando solo el MCD!
p = probabilidad_coprimos(10**6, 200000)
print("Estimación de π:", math.sqrt(6 / p))

Es sorprendente: **π** aparece en un problema que no tiene nada que ver con círculos. Este resultado se debe a Euler y a Cesàro (1881).

**¿Qué pasa si…?**
- …haces el experimento con **tres** números y pides que el MCD de los tres sea 1? ¿La probabilidad sube o baja?

## 6. Aplicaciones: fracciones y mínimo común múltiplo

**Simplificar fracciones:** dividimos numerador y denominador entre su MCD.

**Mínimo común múltiplo:** MCM(a, b) = a · b / MCD(a, b). Sirve, por ejemplo, para saber cuándo vuelven a coincidir dos cosas que se repiten con distintos periodos.

In [ ]:
def simplificar(num, den):
    d = mcd(num, den)
    return num // d, den // d

print("84/60 =", simplificar(84, 60))
print("1071/462 =", simplificar(1071, 462))

def mcm(a, b):
    return a * b // mcd(a, b)

# Un bus pasa cada 12 minutos y otro cada 18. Si salen juntos, ¿cuándo coinciden de nuevo?
print("Vuelven a coincidir a los", mcm(12, 18), "minutos")

---
## ✍️ Ejercicios

Resuelve cada ejercicio en la celda de código que está debajo. Puedes reutilizar `mcd`, `mcm`, `pasos_euclides` y `dibujar_euclides` (si reiniciaste Colab, vuelve a ejecutar las celdas anteriores).

### Ejercicio 1 (básico) · Baldosas
Un patio mide **378 × 238** decímetros.
a) ¿Cuál es la baldosa cuadrada más grande que lo cubre sin cortes?
b) ¿Cuántas baldosas se necesitan?
c) Dibuja el rectángulo con `dibujar_euclides`.

In [ ]:
# Escribe tu código aquí

### Ejercicio 2 (básico) · Suma de fracciones
Calcula 5/12 + 7/18 y da el resultado **simplificado**.
*Pista:* usa el MCM de los denominadores como denominador común, y luego `simplificar`.

In [ ]:
# Escribe tu código aquí

### Ejercicio 3 (intermedio) · Engranajes
Dos engranajes encajan: uno tiene **48 dientes** y el otro **30**. Marcamos con tiza el diente donde se tocan.
a) ¿Cuántas vueltas da cada engranaje antes de que las marcas vuelvan a tocarse?
b) Prueba con 48 y 35 dientes. ¿Por qué los mecánicos prefieren números de dientes coprimos?

In [ ]:
# Escribe tu código aquí

### Ejercicio 4 (intermedio) · Fracciones irreducibles
¿Cuántas fracciones **n/100**, con n entre 1 y 100, ya están simplificadas (es decir, MCD(n, 100) = 1)?
Repite para el denominador 101. ¿Por qué el resultado es tan distinto?

In [ ]:
# Escribe tu código aquí

### Ejercicio 5 (avanzado) · Recursión
Una función **recursiva** se llama a sí misma. La regla MCD(a, b) = MCD(b, a % b) es perfecta para eso.
a) Escribe `mcd_rec(a, b)` que devuelva `a` si `b == 0` y si no devuelva `mcd_rec(b, a % b)`.
b) Comprueba que coincide con `math.gcd` para 1000 parejas al azar.
c) El MCD de **una lista** de números se puede calcular de dos en dos: MCD(a, b, c) = MCD(MCD(a, b), c). Calcula el MCD de [462, 1071, 1309, 2618].

In [ ]:
# Escribe tu código aquí

---
## 🏆 Reto final (opcional) · El acertijo de las jarras

En la película *Duro de matar 3*, los protagonistas tienen una jarra de **5 litros** y otra de **3 litros**, una fuente de agua, y deben medir **exactamente 4 litros**.

1. Resuélvelo a mano: llenar, vaciar o pasar agua de una jarra a otra.
2. Euclides "extendido" dice que con jarras de a y b litros se pueden medir exactamente los múltiplos de **MCD(a, b)** (y nada más). Comprueba: ¿se pueden medir 4 litros con jarras de 6 y 9? ¿Y 3 litros?
3. Escribe un programa que simule las jarras y, probando todas las jugadas posibles, encuentre la solución más corta para medir n litros. *Pista:* guarda cada estado como una pareja `(litros_en_A, litros_en_B)` y usa un `set` para no repetir estados.

In [ ]:
# Escribe tu código aquí

---
## 🎯 Cierre: lo que aprendiste hoy

- El **MCD(a, b)** es el mayor número que divide a los dos; resuelve el problema de la baldosa más grande.
- **Euclides** usa que MCD(a, b) = MCD(b, a % b): los números se achican rápido y el algoritmo funciona incluso con números de cientos de cifras.
- Geométricamente, Euclides **llena un rectángulo con cuadrados**; el último cuadrado es la baldosa del MCD.
- El peor caso de Euclides son los números de **Fibonacci** consecutivos (Lamé, 1844).
- Dos números al azar son coprimos con probabilidad **6/π² ≈ 61 %**.
- El MCD sirve para **simplificar fracciones** y calcular el **MCM** (buses, engranajes).
- En Python aprendiste: **asignación doble** `a, b = b, a % b`, mapas de calor con `imshow`, `math.gcd`, `random.randint` y una primera idea de **recursión**.

**Para pensar:** el algoritmo de Euclides tiene más de 2000 años y se sigue usando en computadoras modernas. ¿Qué otro "algoritmo" conoces que uses sin pensarlo (al sumar, al ordenar cartas, al buscar en un diccionario)?